In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
from sklearn.utils.class_weight import compute_sample_weight

In [2]:
# =====================================================
# 1. 데이터 로드
# =====================================================
df = pd.read_csv('extracted_data.csv')
df = df[df['churn'].notna()].copy()

print('전체 데이터 수:', len(df))

In [3]:
# =====================================================
# 2. 파생 변수 생성 Transformer
# =====================================================
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self, eps=1e-6):
        self.eps = eps

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()

        # 패널 피처 계산 전 정렬
        df = df.sort_values(['id', 'year']).reset_index(drop=True)

        # =========================
        # 기본 비율/차이 피처
        # =========================

        df['income_zero_flag'] = np.where(
            df['income'].notna(),
            (df['income'] == 1).astype('Int64'),
            pd.NA
        )

        df['installment_ratio'] = (
            df['monthly_installment'] /
            (df['monthly_total_cost'] + self.eps)
        )  # 할부금 비율

        income_proxy = df['income'] * 500000 - 750000

        df['cost_income_ratio'] = np.where(
            df['income_zero_flag'] == 0,
            df['monthly_total_cost'] * 1000 / income_proxy,
            np.nan
        )  # 소득 대비 통신비 비중

        df['installment_income_ratio'] = np.where(
            df['income_zero_flag'] == 0,
            df['monthly_installment'] * 1000 / income_proxy,
            np.nan
        )  # 소득 대비 할부금 비중

        df['income_per_person'] = (
            income_proxy.clip(lower=0) / df['household_size']
        )  # 1인당 소득

        df['cost_per_person'] = (
            df['monthly_total_cost'] * 1000 / df['household_size']
        )  # 1인당 통신비

        df['installment_per_person'] = (
            df['monthly_installment'] * 1000 / df['household_size']
        )  # 가구원 1인당 할부금

        df['service_cost'] = (
            df['monthly_total_cost'] - df['monthly_installment']
        )  # 월 총 비용에서 할부금을 뺀 순수 이용요금 성격의 금액

        df['disposable_income_proxy'] = np.where(
            df['income_zero_flag'] == 0,
            income_proxy - df['monthly_total_cost'] * 1000,
            np.nan
        )  # 소득에서 통신비를 뺀 가처분 소득 성격의 값

        df['income_remaining_ratio'] = (
            df['disposable_income_proxy'] / income_proxy
        )  # 소득 대비 남는 비율

        df["service_cost_ratio"] = (
            df["service_cost"] / (df["monthly_total_cost"] + self.eps)
        )  # 총비용 대비 할부 제외 서비스 요금 비중

        df['married_large_family'] = (
            (df['marriage'] == 2) &
            (df['household_size'] >= 3)
        ).astype(int)  # 결혼 + 가구원수

        # 소득과 비용의 분포가 치우친 경우를 완화
        df['income_log'] = np.log1p(df['income'])
        df['monthly_total_cost_log'] = np.log1p(df['monthly_total_cost'])
        df['monthly_installment_log'] = np.log1p(df['monthly_installment'])

        # =========================
        # 패널 데이터 기반 피처
        #   id + year 순으로 정렬 후 사용
        # =========================

        # 직전 연도 데이터 존재 여부 플래그
        # 1 = 직전 연도 데이터 없음
        # 0 = 직전 연도 데이터 있음
        df['is_first_observation'] = df.groupby('id')['year'].shift(1).isna().astype(int)

        # 직전 연도 값
        df['prev_income'] = df.groupby('id')['income'].shift(1)
        df['prev_monthly_total_cost'] = df.groupby('id')['monthly_total_cost'].shift(1)
        df['prev_monthly_installment'] = df.groupby('id')['monthly_installment'].shift(1)
        df['prev_provider'] = df.groupby('id')['provider'].shift(1)

        df['prev_provider'] = df['prev_provider'].fillna(df['provider'])

        # 전년 대비 변화량
        df['income_change'] = df['income'] - df['prev_income']
        df['cost_change'] = df['monthly_total_cost'] - df['prev_monthly_total_cost']
        df['installment_change'] = df['monthly_installment'] - df['prev_monthly_installment']

        # 전년 대비 변화율
        df['income_change_rate'] = df['income_change'] / (df['prev_income'] + self.eps)
        df['cost_change_rate'] = df['cost_change'] / (df['prev_monthly_total_cost'] + self.eps)
        df['installment_change_rate'] = df['installment_change'] / (df['prev_monthly_installment'] + self.eps)

        # 전년 대비 통신사 변경 여부
        df['provider_changed'] = np.where(
            df['prev_provider'].isna(),
            0,
            (df['provider'] != df['prev_provider']).astype(int)
        )

        # 직전 연도 대비 총비용 급증 여부
        df['cost_jump_flag'] = np.where(
            df['is_first_observation'] == 1,
            0,
            (df['cost_change_rate'] >= 0.2).astype(int)
        )

        # 직전 연도 대비 할부금 급증 여부
        df['installment_jump_flag'] = np.where(
            df['is_first_observation'] == 1,
            0,
            (df['installment_change_rate'] >= 0.2).astype(int)
        )

        # =========================
        # 이동통계 피처
        # =========================

        # 최근 3개 시점 평균 통신비
        df['cost_roll_mean_3'] = df.groupby('id')['monthly_total_cost'].transform(
            lambda s: s.shift(1).rolling(3, min_periods=1).mean()
        )

        # 최근 3개 시점 표준편차
        df['cost_roll_std_3'] = df.groupby('id')['monthly_total_cost'].transform(
            lambda s: s.shift(1).rolling(3, min_periods=2).std()
        )

        # 최근 3개 시점 평균 소득
        df['income_roll_mean_3'] = df.groupby('id')['income'].transform(
            lambda s: s.shift(1).rolling(3, min_periods=1).mean()
        )

        # 최근 3개 시점 평균 소득
        df['income_roll_std_3'] = df.groupby('id')['income'].transform(
            lambda s: s.shift(1).rolling(3, min_periods=2).mean()
        )

        # 현재 값이 과거 평균보다 높은지 여부
        df['cost_above_history_mean'] = np.where(
            df['cost_roll_mean_3'].isna(),
            0,
            (df['monthly_total_cost'] > df['cost_roll_mean_3']).astype(int)
        )


        # 결측 제거
        df['income_zero_flag'] = df['income_zero_flag'].fillna(1).astype(int)

In [4]:
# # ----------------------------------
# # 파생변수 생성
# # ----------------------------------
#
# eps = 1e-6
#
# # =========================
# # 기본 비율/차이 피처
# # =========================
#
# df['income_zero_flag'] = np.where(
#     df['income'].notna(),
#     (df['income'] == 1).astype('Int64'),
#     pd.NA
# )
#
# df['installment_ratio'] = (
#     df['monthly_installment'] /
#     (df['monthly_total_cost'] + eps)
# )  # 할부금 비율
#
# df['cost_income_ratio'] = np.where(
#     df['income_zero_flag'] == 0,
#     df['monthly_total_cost'] * 1000 / (df['income'] * 500000 - 750000),
#     np.nan
# )  # 소득 대비 통신비 비중
#
# df['installment_income_ratio'] = np.where(
#     df['income_zero_flag'] == 0,
#     df['monthly_installment'] * 1000 / (df['income'] * 500000 - 750000),
#     np.nan
# )  # 소득 대비 할부금 비중
#
# df['income_per_person'] = (
#     (df['income'] * 500000 - 750000).clip(lower=0) / df['household_size']
# )  # 1인당 소득
#
# df['cost_per_person'] = (
#     df['monthly_total_cost'] * 1000 / df['household_size']
# )  # 1인당 통신비
#
# df['installment_per_person'] = (
#     df['monthly_installment'] * 1000 / df['household_size']
# )  # 가구원 1인당 할부금
#
# df['service_cost'] = (
#     df['monthly_total_cost'] - df['monthly_installment']
# )  # 월 총 비용에서 할부금을 뺀 순수 이용요금 성격의 금액
#
# df['disposable_income_proxy'] = np.where(
#     df['income_zero_flag'] == 0,
#     (df['income'] * 500000 - 750000) - df['monthly_total_cost'] * 1000,
#     np.nan
# )  # 소득에서 통신비를 뺀 가처분 소득 성격의 값
#
# df['income_remaining_ratio'] = (
#     df['disposable_income_proxy'] / (df['income'] * 500000 - 750000)
# )  # 소득 대비 남는 비율
#
# df["service_cost_ratio"] = (
#     df["service_cost"] / (df["monthly_total_cost"] + eps)
# )  # 총비용 대비 할부 제외 서비스 요금 비중
#
# df['married_large_family'] = (
#     (df['marriage'] == 2) &
#     (df['household_size'] >= 3)
# ).astype(int)  # 결혼 + 가구원수
#
# # 소득과 비용의 분포가 치우친 경우를 완화
# df['income_log'] = np.log1p(df['income'])
# df['monthly_total_cost_log'] = np.log1p(df['monthly_total_cost'])
# df['monthly_installment_log'] = np.log1p(df['monthly_installment'])
#
# # =========================
# # 패널 데이터 기반 피처
# #   id + year 순으로 정렬 후 사용
# # =========================
#
# # 직전 연도 데이터 존재 여부 플래그
# # 1 = 직전 연도 데이터 없음
# # 0 = 직전 연도 데이터 있음
# df['is_first_observation'] = df.groupby('id')['year'].shift(1).isna().astype(int)
#
# # 직전 연도 값
# df['prev_income'] = df.groupby('id')['income'].shift(1)
# df['prev_monthly_total_cost'] = df.groupby('id')['monthly_total_cost'].shift(1)
# df['prev_monthly_installment'] = df.groupby('id')['monthly_installment'].shift(1)
# df['prev_provider'] = df.groupby('id')['provider'].shift(1)
#
# df['prev_provider'] = df['prev_provider'].fillna(df['provider'])
#
# # 전년 대비 변화량
# df['income_change'] = df['income'] - df['prev_income']
# df['cost_change'] = df['monthly_total_cost'] - df['prev_monthly_total_cost']
# df['installment_change'] = df['monthly_installment'] - df['prev_monthly_installment']
#
# # 전년 대비 변화율
# df['income_change_rate'] = df['income_change'] / (df['prev_income'] + eps)
# df['cost_change_rate'] = df['cost_change'] / (df['prev_monthly_total_cost'] + eps)
# df['installment_change_rate'] = df['installment_change'] / (df['prev_monthly_installment'] + eps)
#
# # 전년 대비 통신사 변경 여부
# df['provider_changed'] = np.where(
#     df['prev_provider'].isna(),
#     0,
#     (df['provider'] != df['prev_provider']).astype(int)
# )
#
# # 직전 연도 대비 총비용 급증 여부
# df['cost_jump_flag'] = np.where(
#     df['is_first_observation'] == 1,
#     0,
#     (df['cost_change_rate'] >= 0.2).astype(int)
# )
#
# # 직전 연도 대비 할부금 급증 여부
# df['installment_jump_flag'] = np.where(
#     df['is_first_observation'] == 1,
#     0,
#     (df['installment_change_rate'] >= 0.2).astype(int)
# )
#
# # =========================
# # 이동통계 피처
# # =========================
#
# # 최근 3개 시점 평균 통신비
# df['cost_roll_mean_3'] = df.groupby('id')['monthly_total_cost'].transform(
#     lambda s: s.shift(1).rolling(3, min_periods=1).mean()
# )
#
# # 최근 3개 시점 표준편차
# df['cost_roll_std_3'] = df.groupby('id')['monthly_total_cost'].transform(
#     lambda s: s.shift(1).rolling(3, min_periods=2).std()
# )
#
# # 최근 3개 시점 평균 소득
# df['income_roll_mean_3'] = df.groupby('id')['income'].transform(
#     lambda s: s.shift(1).rolling(3, min_periods=1).mean()
# )
#
# # 최근 3개 시점 평균 소득
# df['income_roll_std_3'] = df.groupby('id')['income'].transform(
#     lambda s: s.shift(1).rolling(3, min_periods=2).mean()
# )
#
# # 현재 값이 과거 평균보다 높은지 여부
# df['cost_above_history_mean'] = np.where(
#     df['cost_roll_mean_3'].isna(),
#     0,
#     (df['monthly_total_cost'] > df['cost_roll_mean_3']).astype(int)
# )
#
#
# # 결측 제거
# df['income_zero_flag'] = df['income_zero_flag'].fillna(1).astype(int)

In [5]:
# ----------------------------------
# 2. 변수 구분
# ----------------------------------

numeric_features = [
    'year',
    'age',
    'income',
    'monthly_total_cost',
    'monthly_installment',
    'household_size',
    'installment_ratio',
    'cost_income_ratio',
    'installment_income_ratio',
    'income_per_person',
    'cost_per_person',
    'installment_per_person',
    'service_cost',
    'disposable_income_proxy',
    'income_remaining_ratio',
    'service_cost_ratio',
    'income_log',
    'monthly_total_cost_log',
    'monthly_installment_log',
    'prev_income',
    'prev_monthly_total_cost',
    'prev_monthly_installment',
    'income_change',
    'cost_change',
    'installment_change',
    'income_change_rate',
    'cost_change_rate',
    'installment_change_rate',
    'cost_roll_mean_3',
    'cost_roll_std_3',
    'income_roll_mean_3',
    'income_roll_std_3',
]

categorical_features = [
    'gender',
    'school',
    'job',
    'marriage',
    'cost_payer',
    'provider',
    'married_large_family',
    'is_mobile_bundled',
    'income_zero_flag',
    'is_first_observation',
    'prev_provider',
    'provider_changed',
    'cost_jump_flag',
    'installment_jump_flag',
    'cost_above_history_mean',
]

FEATURE_COLS = numeric_features + categorical_features

# ----------------------------------
# 3. 수치형 파이프라인
# ----------------------------------

numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]
)

# ----------------------------------
# 4. 범주형 파이프라인
# ----------------------------------

categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]
)

# ----------------------------------
# 5. 전체 전처리
# ----------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

In [6]:
def make_full_pipeline(model):
    return Pipeline(
        steps=[
            ('feature_engineering', FeatureEngineer()),
            ('preprocessing', preprocessor),
            ('model', clone(model))
        ]
    )

In [7]:
# --------------------------
# 6. 연도 기준 분할
# --------------------------

train_df = df[df['year'] <= 20]

valid_df = df[
    (df['year'] >= 21) &
    (df['year'] <= 22)
]

test_df = df[df['year'] >= 23]

print(len(train_df))
print(len(valid_df))
print(len(test_df))

91378
18823
8135


In [8]:
# # --------------------------
# # 7. 전처리 적용
# # --------------------------
#
# X_train = train_df.drop(columns=['id', 'churn'])
# y_train = train_df['churn']
#
# X_valid = valid_df.drop(columns=['id', 'churn'])
# y_valid = valid_df['churn']
#
# X_test = test_df.drop(columns=['id', 'churn'])
# y_test = test_df['churn']
#
# X_train = preprocessor.fit_transform(X_train)
# X_valid = preprocessor.transform(X_valid)
# X_test  = preprocessor.transform(X_test)

In [9]:
# # --------------------------
# # 평가 함수 정의
# # --------------------------
#
# def evaluate_model(model, X, y, dataset_name):
#     y_pred = model.predict(X)
#
#     if hasattr(model, 'predict_proba'):
#         y_prob = model.predict_proba(X)[:, 1]
#     else:
#         y_prob = model.predict(X)
#
#     print(f'\n[{dataset_name}]')
#     print('Accuracy :', round(accuracy_score(y, y_pred), 4))
#     print('Precision:', round(precision_score(y, y_pred, zero_division=0), 4))
#     print('Recall   :', round(recall_score(y, y_pred, zero_division=0), 4))
#     print('F1-score :', round(f1_score(y, y_pred, zero_division=0), 4))
#     print('ROC-AUC  :', round(roc_auc_score(y, y_prob), 4))
#
#
# def get_feature_importance(model, preprocessor, top_n=20):
#     feature_names = preprocessor.get_feature_names_out()
#
#     importance_df = pd.DataFrame({
#         'feature'   : feature_names,
#         'importance': model.feature_importances_
#     })
#
#     importance_df = importance_df.sort_values(
#         by='importance',
#         ascending=False
#     ).reset_index(drop=True)
#
#     print(importance_df.head(top_n))
#     return importance_df

In [10]:
# =====================================================
# 하이퍼파라미터 설정 (실험할 때 여기만 수정하세요)
# =====================================================

# --------------------------
# Logistic Regression
# C: 규제 강도 역수. 작을수록 규제 강함
#    후보: 0.01 / 0.1 / 1.0 / 10.0
# --------------------------
LR_C        = 1.0
LR_MAX_ITER = 1000

# --------------------------
# Random Forest
# n_estimators    : 트리 수              후보: 100 / 300 / 500
# max_depth       : None = 제한없음      후보: 5 / 8 / 10 / 15 / None
# min_samples_leaf: 클수록 과적합 방지   후보: 1 / 3 / 5 / 10
# --------------------------
RF_N_ESTIMATORS     = 300
RF_MAX_DEPTH        = 10
RF_MIN_SAMPLES_LEAF = 5

# --------------------------
# Gradient Boosting
# subsample < 1.0 이면 Stochastic GBM (과적합 방지)
# learning_rate 낮추면 n_estimators도 늘려야 함
# --------------------------
GB_N_ESTIMATORS  = 300
GB_MAX_DEPTH     = 5
GB_LEARNING_RATE = 0.05
GB_SUBSAMPLE     = 0.8

# --------------------------
# LightGBM
# num_leaves: max_depth보다 중요   후보: 15 / 31 / 63 / 127
# ※ num_leaves > 2^max_depth 이면 과적합
# --------------------------
LGB_N_ESTIMATORS      = 1000
LGB_LEARNING_RATE     = 0.02
LGB_NUM_LEAVES        = 63
LGB_MIN_CHILD_SAMPLES = 100
LGB_SUBSAMPLE         = 0.8
LGB_COLSAMPLE         = 0.8
LGB_REG_ALPHA         = 1.0
LGB_REG_LAMBDA        = 1.0

# --------------------------
# XGBoost
# scale_pos_weight: 비이탈 수 / 이탈 수   후보: 1 / 3 / 5 / 7
# min_child_weight: 클수록 과적합 방지    후보: 1 / 5 / 10 / 20
# gamma           : 분기 최소 손실 감소   후보: 0 / 0.5 / 1 / 5
# --------------------------
XGB_N_ESTIMATORS     = 1000
XGB_LEARNING_RATE    = 0.03
XGB_MAX_DEPTH        = 4
XGB_MIN_CHILD_WEIGHT = 10
XGB_SUBSAMPLE        = 0.8
XGB_COLSAMPLE        = 0.8
XGB_GAMMA            = 1
XGB_REG_ALPHA        = 1
XGB_REG_LAMBDA       = 2

In [11]:
lr_estimator = LogisticRegression(
    C=LR_C,
    max_iter=LR_MAX_ITER,
    class_weight='balanced',
    random_state=42
)

rf_estimator = RandomForestClassifier(
    n_estimators=RF_N_ESTIMATORS,
    max_depth=RF_MAX_DEPTH,
    min_samples_leaf=RF_MIN_SAMPLES_LEAF,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)

gb_estimator = GradientBoostingClassifier(
    n_estimators=GB_N_ESTIMATORS,
    learning_rate=GB_LEARNING_RATE,
    subsample=GB_SUBSAMPLE,
    random_state=42
)

lgb_estimator = LGBMClassifier(
    objective='binary',
    n_estimators=LGB_N_ESTIMATORS,
    learning_rate=LGB_LEARNING_RATE,
    max_depth=-1,
    num_leaves=LGB_NUM_LEAVES,
    min_child_samples=LGB_MIN_CHILD_SAMPLES,
    subsample=LGB_SUBSAMPLE,
    colsample_bytree=LGB_COLSAMPLE,
    reg_alpha=LGB_REG_ALPHA,
    reg_lambda=LGB_REG_LAMBDA,
    class_weight='balanced',
    random_state=42,
    verbose=-1
)

neg_count = np.sum(train_df['churn'] == 0)
pos_count = np.sum(train_df['churn'] == 1)
XGB_SCALE_POS_WEIGHT = neg_count / pos_count

xgb_estimator = XGBClassifier(
    n_estimators=XGB_N_ESTIMATORS,
    learning_rate=XGB_LEARNING_RATE,
    max_depth=XGB_MAX_DEPTH,
    min_child_weight=XGB_MIN_CHILD_WEIGHT,
    subsample=XGB_SUBSAMPLE,
    colsample_bytree=XGB_COLSAMPLE,
    objective='binary:logistic',
    gamma=XGB_GAMMA,
    reg_alpha=XGB_REG_ALPHA,
    reg_lambda=XGB_REG_LAMBDA,
    eval_metric='auc',
    scale_pos_weight=XGB_SCALE_POS_WEIGHT,
    random_state=42
)

lr_pipe = make_full_pipeline(lr_estimator)
rf_pipe = make_full_pipeline(rf_estimator)
gb_pipe = make_full_pipeline(gb_estimator)
lgb_pipe = make_full_pipeline(lgb_estimator)
xgb_pipe = make_full_pipeline(xgb_estimator)

In [12]:
X_train = train_df.drop(columns=['churn']).copy()
y_train = train_df['churn'].copy()

X_valid = valid_df.drop(columns=['churn']).copy()
y_valid = valid_df['churn'].copy()

X_test = test_df.drop(columns=['churn']).copy()
y_test = test_df['churn'].copy()

In [13]:
lr_pipe.fit(X_train, y_train)
rf_pipe.fit(X_train, y_train)
gb_pipe.fit(X_train, y_train)
lgb_pipe.fit(X_train, y_train)
xgb_pipe.fit(X_train, y_train)

ValueError: Expected 2D array, got scalar array instead:
array=None.
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.

In [ ]:
def evaluate_pipeline(model_pipeline, X, y, dataset_name):
    y_pred = model_pipeline.predict(X)

    if hasattr(model_pipeline, 'predict_proba'):
        y_prob = model_pipeline.predict_proba(X)[:, 1]
    else:
        y_prob = y_pred

    print(f'\n[{dataset_name}]')
    print('Accuracy :', round(accuracy_score(y, y_pred), 4))
    print('Precision:', round(precision_score(y, y_pred, zero_division=0), 4))
    print('Recall   :', round(recall_score(y, y_pred, zero_division=0), 4))
    print('F1-score :', round(f1_score(y, y_pred, zero_division=0), 4))
    print('ROC-AUC  :', round(roc_auc_score(y, y_prob), 4))

In [ ]:
evaluate_pipeline(lr_pipe, X_train, y_train, 'LR Train')
evaluate_pipeline(lr_pipe, X_valid, y_valid, 'LR Validation')
evaluate_pipeline(lr_pipe, X_test, y_test, 'LR Test')

evaluate_pipeline(lgb_pipe, X_train, y_train, 'LGB Train')
evaluate_pipeline(lgb_pipe, X_valid, y_valid, 'LGB Validation')
evaluate_pipeline(lgb_pipe, X_test, y_test, 'LGB Test')

In [ ]:
import joblib
import os

os.makedirs("models", exist_ok=True)

joblib.dump(lr_pipe, "models/lr_full_pipeline.joblib")
joblib.dump(rf_pipe, "models/rf_full_pipeline.joblib")
joblib.dump(gb_pipe, "models/gb_full_pipeline.joblib")
joblib.dump(lgb_pipe, "models/lgb_full_pipeline.joblib")
joblib.dump(xgb_pipe, "models/xgb_full_pipeline.joblib")

---
## Logistic Regression

In [9]:
# --------------------------
# Logistic Regression 학습
# --------------------------

lr_model = LogisticRegression(
    C=LR_C,
    max_iter=LR_MAX_ITER,
    class_weight='balanced',
    random_state=42
)

lr_model.fit(X_train, y_train)

print('train finish')

train finish


In [10]:
# --------------------------
# Logistic Regression 평가
# --------------------------

evaluate_model(lr_model, X_train, y_train, 'Train')
evaluate_model(lr_model, X_valid, y_valid, 'Validation')
evaluate_model(lr_model, X_test,  y_test,  'Test')


[Train]
Accuracy : 0.6063
Precision: 0.4759
Recall   : 0.5822
F1-score : 0.5237
ROC-AUC  : 0.643

[Validation]
Accuracy : 0.5817
Precision: 0.4505
Recall   : 0.7093
F1-score : 0.551
ROC-AUC  : 0.6591

[Test]
Accuracy : 0.5635
Precision: 0.4555
Recall   : 0.6918
F1-score : 0.5493
ROC-AUC  : 0.6379


---
## Random Forest

In [11]:
# --------------------------
# Random Forest 학습
# --------------------------

rf_model = RandomForestClassifier(
    n_estimators=RF_N_ESTIMATORS,
    max_depth=RF_MAX_DEPTH,
    min_samples_leaf=RF_MIN_SAMPLES_LEAF,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)

rf_model.fit(X_train, y_train)

print('train finish')

train finish


In [12]:
# --------------------------
# Random Forest 평가
# --------------------------

evaluate_model(rf_model, X_train, y_train, 'Train')
evaluate_model(rf_model, X_valid, y_valid, 'Validation')
evaluate_model(rf_model, X_test,  y_test,  'Test')


[Train]
Accuracy : 0.6558
Precision: 0.531
Recall   : 0.6354
F1-score : 0.5785
ROC-AUC  : 0.7029

[Validation]
Accuracy : 0.6224
Precision: 0.4838
Recall   : 0.645
F1-score : 0.5529
ROC-AUC  : 0.6717

[Test]
Accuracy : 0.6113
Precision: 0.4958
Recall   : 0.6371
F1-score : 0.5576
ROC-AUC  : 0.6529


In [13]:
# --------------------------
# Random Forest 중요 변수 확인
# --------------------------

rf_importance_df = get_feature_importance(rf_model, preprocessor, top_n=20)

                         feature  importance
0      cat__provider_changed_0.0    0.133927
1      cat__provider_changed_1.0    0.120040
2              cat__provider_1.0    0.101887
3                      num__year    0.065363
4         cat__prev_provider_1.0    0.044834
5              cat__provider_3.0    0.035956
6              cat__provider_2.0    0.025480
7          num__cost_roll_mean_3    0.022124
8              num__service_cost    0.020390
9           num__cost_roll_std_3    0.019466
10          num__cost_per_person    0.018533
11  num__prev_monthly_total_cost    0.018113
12       num__monthly_total_cost    0.017616
13   num__monthly_total_cost_log    0.017394
14        cat__prev_provider_3.0    0.016743
15              num__cost_change    0.016742
16        cat__prev_provider_2.0    0.016242
17  num__disposable_income_proxy    0.015949
18  num__monthly_installment_log    0.015333
19      num__monthly_installment    0.015270


---
## Gradient Boosting

In [14]:
# --------------------------
# Gradient Boosting 학습
# --------------------------

gb_sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

gb_model = GradientBoostingClassifier(
    n_estimators=GB_N_ESTIMATORS,
    max_depth=GB_MAX_DEPTH,
    learning_rate=GB_LEARNING_RATE,
    subsample=GB_SUBSAMPLE,
    random_state=42
)

gb_model.fit(X_train, y_train, sample_weight=gb_sample_weights)

print('train finish')

train finish


In [15]:
# --------------------------
# Gradient Boosting 평가
# --------------------------

evaluate_model(gb_model, X_train, y_train, 'Train')
evaluate_model(gb_model, X_valid, y_valid, 'Validation')
evaluate_model(gb_model, X_test,  y_test,  'Test')


[Train]
Accuracy : 0.6595
Precision: 0.5363
Recall   : 0.6213
F1-score : 0.5757
ROC-AUC  : 0.7123

[Validation]
Accuracy : 0.6446
Precision: 0.5077
Recall   : 0.5873
F1-score : 0.5447
ROC-AUC  : 0.6768

[Test]
Accuracy : 0.627
Precision: 0.5136
Recall   : 0.5675
F1-score : 0.5392
ROC-AUC  : 0.6601


In [16]:
# --------------------------
# Gradient Boosting 중요 변수 확인
# --------------------------

gb_importance_df = get_feature_importance(gb_model, preprocessor, top_n=20)

                          feature  importance
0       cat__provider_changed_0.0    0.247822
1                       num__year    0.100902
2               cat__provider_1.0    0.081232
3          cat__prev_provider_1.0    0.050095
4           num__cost_roll_mean_3    0.039230
5            num__cost_roll_std_3    0.039028
6                        num__age    0.031240
7            num__cost_per_person    0.031096
8               num__service_cost    0.029129
9                num__cost_change    0.025470
10   num__prev_monthly_total_cost    0.025068
11   num__disposable_income_proxy    0.024850
12  num__prev_monthly_installment    0.023323
13        num__installment_change    0.020200
14    num__installment_per_person    0.015752
15       num__monthly_installment    0.013115
16  num__installment_income_ratio    0.012679
17         num__cost_income_ratio    0.012582
18    num__income_remaining_ratio    0.011634
19   num__monthly_installment_log    0.011382


---
## LightGBM

In [17]:
# --------------------------
# LightGBM 학습
# --------------------------

lgb_model = LGBMClassifier(
    objective='binary',
    n_estimators=LGB_N_ESTIMATORS,
    learning_rate=LGB_LEARNING_RATE,
    max_depth=-1,
    num_leaves=LGB_NUM_LEAVES,
    min_child_samples=LGB_MIN_CHILD_SAMPLES,
    subsample=LGB_SUBSAMPLE,
    colsample_bytree=LGB_COLSAMPLE,
    reg_alpha=LGB_REG_ALPHA,
    reg_lambda=LGB_REG_LAMBDA,
    class_weight='balanced',
    random_state=42,
    verbose=-1
)

lgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)]
)

print('train finish')

train finish


In [18]:
# --------------------------
# LightGBM 평가
# --------------------------

evaluate_model(lgb_model, X_train, y_train, 'Train')
evaluate_model(lgb_model, X_valid, y_valid, 'Validation')
evaluate_model(lgb_model, X_test,  y_test,  'Test')


[Train]
Accuracy : 0.7114
Precision: 0.5966
Recall   : 0.6904
F1-score : 0.6401
ROC-AUC  : 0.782

[Validation]
Accuracy : 0.6416
Precision: 0.5042
Recall   : 0.5878
F1-score : 0.5428
ROC-AUC  : 0.6751

[Test]
Accuracy : 0.6227
Precision: 0.5085
Recall   : 0.5655
F1-score : 0.5355
ROC-AUC  : 0.6581


In [19]:
# --------------------------
# LightGBM 중요 변수 확인
# --------------------------

lgb_importance_df = get_feature_importance(lgb_model, preprocessor, top_n=20)

                          feature  importance
0            num__cost_roll_std_3        4779
1           num__cost_roll_mean_3        4660
2           num__cost_change_rate        3717
3                       num__year        3584
4            num__cost_per_person        3105
5    num__prev_monthly_total_cost        3032
6                num__cost_change        2549
7               num__service_cost        2547
8         num__monthly_total_cost        2390
9         num__service_cost_ratio        2245
10         num__cost_income_ratio        2205
11   num__disposable_income_proxy        1990
12                       num__age        1937
13  num__prev_monthly_installment        1872
14   num__installment_change_rate        1862
15         num__installment_ratio        1709
16        num__income_roll_mean_3        1683
17         num__income_roll_std_3        1444
18  num__installment_income_ratio        1283
19        num__installment_change        1258


---
## XGBoost

In [20]:
# --------------------------
# XGBoost 학습
# --------------------------

neg_count = np.sum(y_train == 0)
pos_count = np.sum(y_train == 1)
XGB_SCALE_POS_WEIGHT = neg_count / pos_count

xgb_model = XGBClassifier(
    n_estimators=XGB_N_ESTIMATORS,
    learning_rate=XGB_LEARNING_RATE,
    max_depth=XGB_MAX_DEPTH,
    min_child_weight=XGB_MIN_CHILD_WEIGHT,
    subsample=XGB_SUBSAMPLE,
    colsample_bytree=XGB_COLSAMPLE,
    objective='binary:logistic',
    gamma=XGB_GAMMA,
    reg_alpha=XGB_REG_ALPHA,
    reg_lambda=XGB_REG_LAMBDA,
    eval_metric='auc',
    scale_pos_weight=XGB_SCALE_POS_WEIGHT,
    random_state=42
)

xgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=False
)

print('train finish')

train finish


In [21]:
# --------------------------
# XGBoost 평가
# --------------------------

evaluate_model(xgb_model, X_train, y_train, 'Train')
evaluate_model(xgb_model, X_valid, y_valid, 'Validation')
evaluate_model(xgb_model, X_test,  y_test,  'Test')


[Train]
Accuracy : 0.654
Precision: 0.5298
Recall   : 0.6157
F1-score : 0.5695
ROC-AUC  : 0.7036

[Validation]
Accuracy : 0.6425
Precision: 0.5052
Recall   : 0.5935
F1-score : 0.5458
ROC-AUC  : 0.6777

[Test]
Accuracy : 0.6248
Precision: 0.5109
Recall   : 0.5719
F1-score : 0.5397
ROC-AUC  : 0.658


In [22]:
# --------------------------
# XGBoost 중요 변수 확인
# --------------------------

xgb_importance_df = get_feature_importance(xgb_model, preprocessor, top_n=20)

                          feature  importance
0       cat__provider_changed_1.0    0.207434
1       cat__provider_changed_0.0    0.178286
2               cat__provider_1.0    0.092459
3          cat__prev_provider_1.0    0.021544
4               cat__provider_3.0    0.019524
5               cat__provider_2.0    0.016263
6                       num__year    0.015954
7      cat__is_mobile_bundled_2.0    0.015550
8          cat__prev_provider_2.0    0.012249
9   cat__is_first_observation_1.0    0.011673
10              cat__provider_5.0    0.011211
11                       num__age    0.010966
12     cat__is_mobile_bundled_1.0    0.010711
13  cat__is_first_observation_0.0    0.010508
14   num__monthly_installment_log    0.010302
15       num__monthly_installment    0.010031
16              cat__marriage_1.0    0.009676
17         cat__prev_provider_3.0    0.008837
18              cat__provider_4.0    0.007892
19  num__prev_monthly_installment    0.007657


---
## Voting Ensemble

In [23]:
# --------------------------
# Voting Ensemble 학습
# --------------------------
# 여러 모델의 확률 예측을 평균내는 소프트 보팅
# 개별 모델보다 안정적인 성능이 나오는 경우가 많음

voting_model = VotingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(
            n_estimators=RF_N_ESTIMATORS,
            max_depth=RF_MAX_DEPTH,
            class_weight='balanced',
            n_jobs=-1,
            random_state=42
        )),
        ('lgb', LGBMClassifier(
            n_estimators=LGB_N_ESTIMATORS,
            learning_rate=LGB_LEARNING_RATE,
            num_leaves=LGB_NUM_LEAVES,
            class_weight='balanced',
            random_state=42,
            verbose=-1
        )),
        ('xgb', XGBClassifier(
            n_estimators=XGB_N_ESTIMATORS,
            learning_rate=XGB_LEARNING_RATE,
            max_depth=XGB_MAX_DEPTH,
            eval_metric='auc',
            scale_pos_weight=XGB_SCALE_POS_WEIGHT,
            random_state=42,
            verbosity=0
        ))
    ],
    voting='soft',
    n_jobs=-1
)

voting_model.fit(X_train, y_train)

print('train finish')

train finish


In [24]:
# --------------------------
# Voting Ensemble 평가
# --------------------------

evaluate_model(voting_model, X_train, y_train, 'Train')
evaluate_model(voting_model, X_valid, y_valid, 'Validation')
evaluate_model(voting_model, X_test,  y_test,  'Test')


[Train]
Accuracy : 0.6835
Precision: 0.5647
Recall   : 0.6483
F1-score : 0.6036
ROC-AUC  : 0.7483

[Validation]
Accuracy : 0.6416
Precision: 0.504
Recall   : 0.6016
F1-score : 0.5485
ROC-AUC  : 0.6781

[Test]
Accuracy : 0.6289
Precision: 0.5154
Recall   : 0.5847
F1-score : 0.5479
ROC-AUC  : 0.6599
